# Übersicht der Decoding-Ansätze

Dieses Notebook dokumentiert verschiedene Strategien und Entwicklungsstufen zur Dekodierung der QR-Codes.

**Anleitung zur Integration:**
Soll einer dieser Ansätze im `Modeltest.ipynb` angewendet werden, muss der entsprechende Codeblock einfach an die passende Stelle im Ziel-Notebook kopiert werden.

**Hinweis zur finalen Implementierung:**
Standardmäßig ist in `Modeltest.ipynb` bereits der **3. Ansatz** hinterlegt. Für diesen wurde sich final entschieden, da er die **beste Performance** bei gleichzeitig **geringster Laufzeit** aufweist.

#### 1. Ansatz

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from pyzbar.pyzbar import decode, ZBarSymbol

# ==========================================
# 1. KONFIGURATION
# ==========================================
class PostProcessConfig:
    original_img_dir = Path('test_picture')
    heatmap_dir = Path('heatmaps_output')
    
    base_output_dir = Path('decoder/ersteMethode')
    dir_success = base_output_dir / 'success'
    dir_failed = base_output_dir / 'failed'
    log_file = base_output_dir / 'scan_results.txt'
    
    heatmap_threshold_pixel_val = 51  
    padding = 25  

# ==========================================
# 2. HILFSFUNKTIONEN
# ==========================================
def add_quiet_zone(img):
    """
    Fügt einen weißen Rahmen hinzu. Das ist oft der einzige Grund,
    warum ein perfekter Crop nicht gelesen wird.
    """
    border = 30
    return cv2.copyMakeBorder(img, border, border, border, border, cv2.BORDER_CONSTANT, value=[255, 255, 255])

# ==========================================
# 3. DEKODIERUNG (DEIN ORIGINAL + QUIET ZONE)
# ==========================================
def apply_decoding_tricks(roi):
    """
    Dein ursprünglicher 19%-Ansatz, aber mit erzwungener 'Quiet Zone'.
    """
    if roi is None or roi.size == 0: return None

    # SCHRITT 0: Quiet Zone hinzufügen (Das fehlte im Original)
    roi = add_quiet_zone(roi)

    attempts = []
    
    # A: Graustufen (Standard)
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    attempts.append(gray)
    
    # B: Otsu Binarisierung (Starker Kontrast)
    _, binary_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    attempts.append(binary_otsu)
    
    # C: Adaptives Thresholding (Gegen Schatten)
    adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    attempts.append(adaptive)

    # D: Zoom & Schärfen (Deine Logik)
    # 2x Zoom hilft bei kleiner Auflösung enorm
    roi_big = cv2.resize(gray, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    roi_sharp = cv2.filter2D(roi_big, -1, kernel)
    attempts.append(roi_sharp)
    
    # E: Zoom & Binarisierung (Neu kombiniert)
    # Manchmal hilft Zoom + harter Kontrast zusammen
    _, big_binary = cv2.threshold(roi_sharp, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    attempts.append(big_binary)

    # --- SCHLEIFE ---
    for img_variant in attempts:
        # 1. Normaler Versuch
        res = decode(img_variant, symbols=[ZBarSymbol.QRCODE])
        if res: return res[0]
        
        # 2. Rotations-Versuche
        # Wir nutzen deine ursprüngliche Winkel-Liste, die gut funktionierte
        h, w = img_variant.shape
        center = (w // 2, h // 2)
        
        # Winkel erweitert um leichte Schieflagen (10, -10)
        for angle in [10, -10, 30, 45, 60, 90, -30, -45, -60, -90]: 
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            # WICHTIG: borderValue=255 (Weiß) statt Schwarz, damit der Rand erhalten bleibt!
            rotated = cv2.warpAffine(img_variant, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=255)
            
            res_rot = decode(rotated, symbols=[ZBarSymbol.QRCODE])
            if res_rot: return res_rot[0]
            
    return None

# ==========================================
# 4. HAUPTPROGRAMM
# ==========================================
# ==========================================
# 4. HAUPTPROGRAMM (RESTORED - OPTIMIERT)
# ==========================================
def run_decoding_restored():
    cfg = PostProcessConfig()
    cfg.dir_success.mkdir(parents=True, exist_ok=True)
    cfg.dir_failed.mkdir(parents=True, exist_ok=True)
    
    heatmap_files = list(cfg.heatmap_dir.glob("*_heatmap.png"))
    if not heatmap_files:
        print("❌ Keine Heatmaps gefunden!")
        return

    print(f"📂 Starte Analyse (Restored 19% Version + QuietZone)...")
    
    # --- STATISTIK VARIABLEN ---
    stats_total_images = 0
    stats_relevant_images = 0
    stats_success = 0
    stats_failed = 0
    stats_codes = 0

    with open(cfg.log_file, 'w', encoding='utf-8') as log:
        log.write("Dateiname;Status;Inhalt\n") 
        
        for hm_file in heatmap_files:
            stem = hm_file.stem.replace("_heatmap", "")
            
            orig_path = None
            for ext in [".jpg", ".jpeg", ".png", ".JPG", ".PNG"]:
                possible = cfg.original_img_dir / (stem + ext)
                if possible.exists(): orig_path = possible; break
            
            if not orig_path: continue
            
            img_orig = cv2.imread(str(orig_path))
            img_heatmap = cv2.imread(str(hm_file), cv2.IMREAD_GRAYSCALE)
            
            if img_orig is None: continue
            
            stats_total_images += 1
            h_orig, w_orig = img_orig.shape[:2]

            _, thresh = cv2.threshold(img_heatmap, cfg.heatmap_threshold_pixel_val, 255, cv2.THRESH_BINARY)
            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # --- FILTERUNG: Nur relevante Bilder zählen ---
            if len(contours) == 0:
                continue
            
            stats_relevant_images += 1

            result_img = img_orig.copy()
            image_has_code = False
            found_contents = []

            for cnt in contours:
                x, y, w, h = cv2.boundingRect(cnt)
                if w < 20 or h < 20: continue 
                
                # ROI Cut
                x_pad = max(0, x - cfg.padding)
                y_pad = max(0, y - cfg.padding)
                w_pad = min(w_orig - x_pad, w + 2*cfg.padding)
                h_pad = min(h_orig - y_pad, h + 2*cfg.padding)
                roi = img_orig[y_pad : y_pad + h_pad, x_pad : x_pad + w_pad]
                
                decoded_obj = apply_decoding_tricks(roi)

                if decoded_obj:
                    image_has_code = True
                    content = decoded_obj.data.decode("utf-8")
                    found_contents.append(content)
                    stats_codes += 1
                    
                    print(f"   ✅ {stem}: {content}")
                    cv2.rectangle(result_img, (x, y), (x + w, y + h), (0, 255, 0), 4)
                    cv2.putText(result_img, "OK", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    # Hier könnte man auch breaken, wenn pro Bild nur 1 Code erwartet wird
                else:
                    cv2.rectangle(result_img, (x, y), (x + w, y + h), (0, 0, 255), 2)

            out_name = f"res_{orig_path.name}"
            if image_has_code:
                stats_success += 1
                cv2.imwrite(str(cfg.dir_success / out_name), result_img)
                for c in found_contents:
                    log.write(f"{orig_path.name};OK;{c}\n")
            else:
                stats_failed += 1
                cv2.imwrite(str(cfg.dir_failed / out_name), result_img)
                log.write(f"{orig_path.name};FAILED;-\n")

    print("\n" + "="*50)
    print("             ERGEBNIS (RESTORED)")
    print("="*50)
    
    if stats_relevant_images > 0:
        rate = (stats_success / stats_relevant_images) * 100
    else: rate = 0
    
    print(f"Gesamt Bilder:        {stats_total_images}")
    print(f"Relevant (mit Code):  {stats_relevant_images}")
    print("-" * 30)
    print(f"Erfolg:               {stats_success}")
    print(f"Fehlschlag:           {stats_failed}")
    print(f"--> Erfolgsquote:     {rate:.1f}%")
    print("="*50)

if __name__ == "__main__":
    run_decoding_restored()

#### 2. Ansatz

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from pyzbar.pyzbar import decode, ZBarSymbol

# ==========================================
# 1. KONFIGURATION
# ==========================================
class PostProcessConfig:
    original_img_dir = Path('test_picture')
    heatmap_dir = Path('heatmaps_output')
    
    base_output_dir = Path('decoder/hoehere_variantenvielfalt')
    dir_success = base_output_dir / 'success'
    dir_failed = base_output_dir / 'failed'
    log_file = base_output_dir / 'scan_results.txt'
    
    # Dein bevorzugter Wert
    heatmap_threshold_pixel_val = 51  
    padding = 60

# ==========================================
# 2. DER "20%-DECODER" (Dein Sieger-Code)
# ==========================================
def add_quiet_zone(img):
    """Fügt den lebenswichtigen weißen Rand hinzu."""
    border = 30
    return cv2.copyMakeBorder(img, border, border, border, border, cv2.BORDER_CONSTANT, value=[255, 255, 255])

def robust_decode_func(roi_chunk):
    """
    Das ist EXAKT die Logik aus deiner 20%-Lösung.
    Wir haben sie in eine Funktion gepackt, um sie mehrfach anwenden zu können.
    """
    if roi_chunk is None or roi_chunk.size == 0: return None

    # 1. Quiet Zone (Basis für alles)
    roi = add_quiet_zone(roi_chunk)
    
    attempts = []
    
    # A: Graustufen
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    attempts.append(gray)
    
    # B: Otsu
    _, binary_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    attempts.append(binary_otsu)
    
    # C: Adaptive Threshold
    adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    attempts.append(adaptive)

    # D: Zoom & Schärfen (Das fehlte im 16% Versuch!)
    roi_big = cv2.resize(gray, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    roi_sharp = cv2.filter2D(roi_big, -1, kernel)
    attempts.append(roi_sharp)
    
    # E: Zoom & Binarisierung
    _, big_binary = cv2.threshold(roi_sharp, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    attempts.append(big_binary)

    # --- DIE SUCHE ---
    for img_variant in attempts:
        # 1. Scan normal
        res = decode(img_variant, symbols=[ZBarSymbol.QRCODE])
        if res: return res[0]
        
        # 2. Rotation (Nur wenn Bildgröße sinnvoll ist, um Zeit zu sparen)
        h, w = img_variant.shape
        center = (w // 2, h // 2)
        
        # Deine bewährte Winkel-Liste
        for angle in [10, -10, 30, 45, 60, -30, -45, -60]: 
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            rotated = cv2.warpAffine(img_variant, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=255)
            
            res_rot = decode(rotated, symbols=[ZBarSymbol.QRCODE])
            if res_rot: return res_rot[0]
            
    return None

# ==========================================
# 3. DIE INTELLIGENTE SUCHE (Smart Tiling)
# ==========================================
def process_roi_smart(roi):
    """
    Kombiniert den ROI-Scan mit einer gezielten Suche im Zentrum und den Ecken.
    Verhindert, dass Codes durch Sliding-Windows "zerschnitten" werden.
    """
    h, w = roi.shape[:2]

    # --- SCHRITT 1: Der "20%-Versuch" (Ganzes Bild) ---
    # Das garantiert, dass wir nicht schlechter als vorher sind!
    if res := robust_decode_func(roi):
        return res, "Full ROI"

    # Wenn der ROI winzig ist, bringt Aufteilen nichts
    if w < 100 or h < 100: return None, None

    # --- SCHRITT 2: Der "Center Crop" (Gegen riesige rote Boxen) ---
    # Wir schneiden die mittleren 60% aus. Oft liegt der Code dort, 
    # aber der Scheibenwischer am Rand stört den Scanner beim Full-Scan.
    cx, cy = w // 2, h // 2
    cw, ch = int(w * 0.6), int(h * 0.6)
    x_start = cx - cw // 2
    y_start = cy - ch // 2
    
    center_crop = roi[y_start:y_start+ch, x_start:x_start+cw]
    if res := robust_decode_func(center_crop):
        return res, "Center Crop"

    # --- SCHRITT 3: Die 4 Ecken (Gegen dezentrale Codes) ---
    # Wir nehmen 4 große Überlappende Bereiche (jeweils 60% des Bildes)
    # Top-Left, Top-Right, Bottom-Left, Bottom-Right
    crops = [
        (roi[0:ch, 0:cw], "Top-Left"),
        (roi[0:ch, w-cw:w], "Top-Right"),
        (roi[h-ch:h, 0:cw], "Bottom-Left"),
        (roi[h-ch:h, w-cw:w], "Bottom-Right")
    ]
    
    for crop_img, name in crops:
        if res := robust_decode_func(crop_img):
            return res, name

    return None, None

# ==========================================
# 4. HAUPTPROGRAMM
# ==========================================
def run_decoding_combined():
    cfg = PostProcessConfig()
    cfg.dir_success.mkdir(parents=True, exist_ok=True)
    cfg.dir_failed.mkdir(parents=True, exist_ok=True)
    
    heatmap_files = list(cfg.heatmap_dir.glob("*_heatmap.png"))
    if not heatmap_files:
        print("❌ Keine Heatmaps gefunden!")
        return

    print(f"📂 Starte 'Combined Force' Analyse (20%-Logik + Smart Crops) von {len(heatmap_files)} Bildern...")
    
    # --- STATISTIK VARIABLEN ---
    stats_total_images = 0      # Alle Bilder im Ordner
    stats_relevant_images = 0   # Nur Bilder, wo die Heatmap etwas angezeigt hat
    stats_success = 0           # Davon erfolgreich dekodiert
    stats_failed = 0            # Davon fehlgeschlagen
    stats_codes = 0             # Anzahl gefundener Codes total

    with open(cfg.log_file, 'w', encoding='utf-8') as log:
        log.write("Dateiname;Status;Inhalt;Methode\n") 
        
        for hm_file in heatmap_files:
            stem = hm_file.stem.replace("_heatmap", "")
            
            orig_path = None
            for ext in [".jpg", ".jpeg", ".png", ".JPG", ".PNG"]:
                possible = cfg.original_img_dir / (stem + ext)
                if possible.exists(): orig_path = possible; break
            
            if not orig_path: continue
            
            img_orig = cv2.imread(str(orig_path))
            img_heatmap = cv2.imread(str(hm_file), cv2.IMREAD_GRAYSCALE)
            
            if img_orig is None: continue
            
            stats_total_images += 1
            h_orig, w_orig = img_orig.shape[:2]

            _, thresh = cv2.threshold(img_heatmap, cfg.heatmap_threshold_pixel_val, 255, cv2.THRESH_BINARY)
            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # --- FILTERUNG: Nur relevante Bilder zählen ---
            if len(contours) == 0:
                continue
            
            stats_relevant_images += 1

            result_img = img_orig.copy()
            image_has_code = False
            found_contents = []

            for cnt in contours:
                x, y, w, h = cv2.boundingRect(cnt)
                if w < 20 or h < 20: continue 
                
                # ROI Cut
                x_pad = max(0, x - cfg.padding)
                y_pad = max(0, y - cfg.padding)
                w_pad = min(w_orig - x_pad, w + 2*cfg.padding)
                h_pad = min(h_orig - y_pad, h + 2*cfg.padding)
                roi = img_orig[y_pad : y_pad + h_pad, x_pad : x_pad + w_pad]
                
                # --- KOMBINIERTE LOGIK ---
                decoded_obj, method_name = process_roi_smart(roi)

                if decoded_obj:
                    image_has_code = True
                    content = decoded_obj.data.decode("utf-8")
                    found_contents.append(content)
                    stats_codes += 1
                    
                    print(f"   ✅ {stem} [{method_name}]: {content}")
                    cv2.rectangle(result_img, (x, y), (x + w, y + h), (0, 255, 0), 4)
                    cv2.putText(result_img, f"OK ({method_name})", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                    break 
                else:
                    cv2.rectangle(result_img, (x, y), (x + w, y + h), (0, 0, 255), 2)

            out_name = f"res_{orig_path.name}"
            if image_has_code:
                stats_success += 1
                cv2.imwrite(str(cfg.dir_success / out_name), result_img)
                for c in found_contents:
                    log.write(f"{orig_path.name};OK;{c};Success\n")
            else:
                # Bild war relevant (Konturen), aber Decodierung schlug fehl
                stats_failed += 1
                cv2.imwrite(str(cfg.dir_failed / out_name), result_img)
                log.write(f"{orig_path.name};FAILED;-;-\n")

    print("\n" + "="*50)
    print("             ERGEBNIS (COMBINED FORCE)")
    print("="*50)
    
    if stats_relevant_images > 0:
        rate = (stats_success / stats_relevant_images) * 100
    else: rate = 0
    
    print(f"Gesamt Bilder:        {stats_total_images}")
    print(f"Relevant (mit Code):  {stats_relevant_images}")
    print("-" * 30)
    print(f"Erfolg:               {stats_success}")
    print(f"Fehlschlag:           {stats_failed}")
    print(f"--> Erfolgsquote:     {rate:.1f}%")
    print("-" * 30)
    print(f"Gefundene Codes:      {stats_codes}")
    print("="*50)

if __name__ == "__main__":
    run_decoding_combined()

#### 3. Ansatz

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from qreader import QReader

# ==========================================
# 1. KONFIGURATION
# ==========================================
class PostProcessConfig:
    original_img_dir = Path('test_picture')
    heatmap_dir = Path('heatmaps_output')
    
    base_output_dir = Path('decoder/final_results_qreader')
    dir_success = base_output_dir / 'success'
    dir_failed = base_output_dir / 'failed'
    log_file = base_output_dir / 'scan_results.txt'
    
    heatmap_threshold_pixel_val = 51  
    padding = 30  # Padding für Kontext um den QR-Code

# ==========================================
# 2. HAUPTPROGRAMM
# ==========================================
def run_decoding_qreader():
    cfg = PostProcessConfig()
    cfg.dir_success.mkdir(parents=True, exist_ok=True)
    cfg.dir_failed.mkdir(parents=True, exist_ok=True)
    
    heatmap_files = list(cfg.heatmap_dir.glob("*_heatmap.png"))
    if not heatmap_files:
        print("❌ Keine Heatmaps gefunden!")
        return [] # Leere Liste zurückgeben

    print("⏳ Lade QReader AI Modell...")
    # Niedrige Confidence (0.1), um auch verzerrte Codes für die Winkelanalyse zu erfassen
    reader = QReader(model_size='s', min_confidence=0.1)
    print("✅ Modell geladen.")

    print(f"📂 Starte Analyse mit QReader...")
    
    # --- SPEICHER FÜR NACHGELAGERTE PnP-ANALYSE ---
    results_storage = [] 
    
    # Statistik-Zähler
    stats_total_images = 0      
    stats_relevant_images = 0   
    stats_success = 0           
    stats_failed = 0            

    with open(cfg.log_file, 'w', encoding='utf-8') as log:
        log.write("Dateiname;Status;Inhalt\n") 
        
        for hm_file in heatmap_files:
            stem = hm_file.stem.replace("_heatmap", "")
            
            # Originalbild zuordnen
            orig_path = None
            for ext in [".jpg", ".jpeg", ".png", ".JPG", ".PNG"]:
                possible = cfg.original_img_dir / (stem + ext)
                if possible.exists(): orig_path = possible; break
            
            if not orig_path: continue
            
            img_orig = cv2.imread(str(orig_path))
            img_heatmap = cv2.imread(str(hm_file), cv2.IMREAD_GRAYSCALE)
            
            if img_orig is None: continue
            
            stats_total_images += 1
            h_orig, w_orig = img_orig.shape[:2]

            # Heatmap binarisieren und Konturen finden
            _, thresh = cv2.threshold(img_heatmap, cfg.heatmap_threshold_pixel_val, 255, cv2.THRESH_BINARY)
            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # Bilder ohne Heatmap-Treffer überspringen (nicht relevant für Quote)
            if len(contours) == 0:
                continue
            
            stats_relevant_images += 1

            result_img = img_orig.copy()
            image_has_code = False
            found_contents = []

            for cnt in contours:
                x, y, w, h = cv2.boundingRect(cnt)
                # Kleine Störungen ignorieren
                if w < 20 or h < 20: continue 
                
                # ROI mit Padding ausschneiden
                x_pad = max(0, x - cfg.padding)
                y_pad = max(0, y - cfg.padding)
                w_pad = min(w_orig - x_pad, w + 2*cfg.padding)
                h_pad = min(h_orig - y_pad, h + 2*cfg.padding)
                
                roi = img_orig[y_pad : y_pad + h_pad, x_pad : x_pad + w_pad]
                
                # --- QREADER LOGIK & DATENERFASSUNG ---
                if roi is not None and roi.size > 0:
                    # Konvertierung für QReader (erwartet RGB)
                    roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
                    
                    # 1. Detektion (Liefert Geometrie/Polygone)
                    detections = reader.detect(image=roi_rgb)
                    
                    if detections:
                        for detection in detections:
                            # 2. Dekodierung (Liefert Textinhalt basierend auf Detektion)
                            content = reader.decode(image=roi_rgb, detection_result=detection)
                            
                            # Speichern der Daten für Winkelberechnung (PnP)
                            results_storage.append({
                                'filename': orig_path.name,
                                'image': roi.copy(),                 # Kopie des ROI für Visualisierung
                                'polygon': detection['polygon_xy'],  # Eckpunkte für PnP-Berechnung
                                'content': content,
                                'is_readable': content is not None
                            })

                            if content:
                                image_has_code = True
                                found_contents.append(content)
                                print(f"   ✅ {stem}: {content}")
                                
                                # Visualisierung im Ergebnisbild
                                cv2.rectangle(result_img, (x, y), (x + w, y + h), (0, 255, 0), 4)
                                short = (content[:15] + '..') if len(content) > 15 else content
                                cv2.putText(result_img, short, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    
                    # Fallback-Visualisierung: Heatmap positiv, aber kein Text lesbar
                    if not image_has_code:
                         cv2.rectangle(result_img, (x, y), (x + w, y + h), (0, 0, 255), 2)

            # Ergebnisbilder und Logs speichern
            out_name = f"res_{orig_path.name}"
            
            if image_has_code:
                stats_success += 1
                cv2.imwrite(str(cfg.dir_success / out_name), result_img)
                for c in found_contents:
                    log.write(f"{orig_path.name};OK;{c}\n")
            else:
                stats_failed += 1
                cv2.imwrite(str(cfg.dir_failed / out_name), result_img)
                log.write(f"{orig_path.name};FAILED;-\n")

    # Statistik Ausgabe
    print("\n" + "="*50)
    print("             ERGEBNIS (QREADER)")
    print("="*50)
    
    if stats_relevant_images > 0:
        rate = (stats_success / stats_relevant_images) * 100
    else: 
        rate = 0
        
    print(f"Gesamt Bilder im Ordner:  {stats_total_images}")
    print(f"Davon mit Code-Verdacht:  {stats_relevant_images} (Basis für Quote)")
    print("-" * 30)
    print(f"Erfolgreich dekodiert:    {stats_success}")
    print(f"Nicht lesbar (trotz ROI): {stats_failed}")
    print(f"--> Erfolgsquote:         {rate:.1f}%")
    print("="*50)

    # Rückgabe der gesammelten Daten für den nächsten Codeblock
    return results_storage

if __name__ == "__main__":
    # Ergebnis in Variable speichern für den nachfolgenden PnP-Block
    collected_data = run_decoding_qreader()